# EHRSHOT sequence baseline: numeric-aware embeddings

Цель: взять готовые sequence datasets из предыдущего ноутбука (`raw`, `compressed_dedup`, `compressed_first_last`, `compressed_condition_era`, `compressed_hybrid`) и проверить, помогает ли числовой сигнал (`numeric_values`) внутри sequence-моделей.

Что меняем относительно code-only baseline:

1. `numeric_values` используются в event embedding.
2. Числа нормализуются **по token/code** на train split, чтобы не смешивать разные шкалы лабораторных анализов, дозировок и compression duration.
3. Для GRU/LSTM/RETAIN время кодируется через `Time2Vec`-style encoder.
4. Метрики: AUROC, AUPRC, Brier, LogLoss, top-k lift/event capture.

В code-only sequence baseline каждое событие кодировалось только через `token_id` и временные признаки. В этом ноутбуке event embedding расширен:

```text
event_embedding =
    token_embedding
  + time_embedding(log1p(days_before_prediction), log1p(delta_days))
  + numeric_embedding(z_value, numeric_present)
```

`numeric_value` нормализуется отдельно для каждого token/code по train split. Это нужно, потому что разные коды имеют разные шкалы: лабораторные значения, дозировки, duration у compression tokens и т.д.

## 0. Imports and config

In [1]:
from pathlib import Path
import json
import math
import random
import zipfile
import warnings
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss
from sklearn.linear_model import LogisticRegression

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("DEVICE:", DEVICE)

PAD_ID = 0
UNK_ID = 1

DEVICE: mps


## 1. Где лежат готовые sequence datasets

In [2]:
# Если datasets уже лежат рядом с ноутбуком:
USE_LOCAL_SEQUENCE_DATASETS = True
LOCAL_SEQUENCE_DATA_DIR = Path("ehrshot_sequence_datasets")

# Если datasets были загружены как artifact в ClearML:
USE_CLEARML_DOWNLOAD = False
CLEARML_SOURCE_TASK_ID = ""  # например: "84e2aef056324facb0d0dcc12baa9562"
CLEARML_SEQUENCE_ARTIFACT_CANDIDATES = ["sequence_datasets", "sequence_data_dir", "ehrshot_sequence_datasets"]
DOWNLOAD_DIR = Path("clearml_downloads")

RESULTS_DIR = Path("ehrshot_sequence_numeric_results")
CHECKPOINT_DIR = RESULTS_DIR / "checkpoints"
for p in [DOWNLOAD_DIR, RESULTS_DIR, CHECKPOINT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SELECTED_TASKS = ["guo_readmission", "guo_icu"]
SEQUENCE_VERSIONS = [
    "raw",
    "compressed_dedup",
    "compressed_first_last",
    "compressed_condition_era",
    "compressed_hybrid",
]

DEBUG_RUN = False
if DEBUG_RUN:
    RUN_TASKS = ["guo_readmission"]
    RUN_SEQUENCE_VERSIONS = ["raw", "compressed_dedup"]
    RUN_MODEL_NAMES = ["GRU_1L_numeric", "RETAIN_lite_numeric"]
    MAX_TRAIN_EXAMPLES = 800
    MAX_EVAL_EXAMPLES = 400
    N_EPOCHS = 2
else:
    RUN_TASKS = SELECTED_TASKS
    RUN_SEQUENCE_VERSIONS = SEQUENCE_VERSIONS
    RUN_MODEL_NAMES = [
        "GRU_1L_numeric",
        "GRU_2L_numeric",
        "LSTM_1L_numeric",
        "LSTM_2L_numeric",
        "RETAIN_lite_numeric",
    ]
    MAX_TRAIN_EXAMPLES = None
    MAX_EVAL_EXAMPLES = None
    N_EPOCHS = 12

MAX_LEN = 1024
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 3
GRAD_CLIP = 1.0
NUM_WORKERS = 0
SAVE_MODEL_CHECKPOINTS = True

In [3]:
def download_sequence_artifact_from_clearml() -> Path:
    if not CLEARML_SOURCE_TASK_ID:
        raise ValueError("Set CLEARML_SOURCE_TASK_ID before using ClearML download.")
    from clearml import Task
    source_task = Task.get_task(task_id=CLEARML_SOURCE_TASK_ID)
    available = list(source_task.artifacts.keys())
    print("Available artifacts:", available)

    artifact = None
    artifact_name = None
    for name in CLEARML_SEQUENCE_ARTIFACT_CANDIDATES:
        if name in source_task.artifacts:
            artifact = source_task.artifacts[name]
            artifact_name = name
            break
    if artifact is None:
        raise KeyError(f"No sequence artifact found. Available: {available}")

    local_copy = Path(artifact.get_local_copy())
    print(f"Downloaded artifact {artifact_name} to:", local_copy)

    out_dir = DOWNLOAD_DIR / artifact_name
    out_dir.mkdir(parents=True, exist_ok=True)
    if local_copy.is_file() and local_copy.suffix.lower() == ".zip":
        with zipfile.ZipFile(local_copy, "r") as zf:
            zf.extractall(out_dir)
        return out_dir
    if local_copy.is_dir():
        return local_copy
    return local_copy.parent


def looks_like_sequence_root(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False
    for task_name in SELECTED_TASKS:
        for version in SEQUENCE_VERSIONS:
            if (path / task_name / version / "examples.parquet").exists():
                return True
    return False


def find_sequence_dataset_root(base: Path) -> Path:
    candidates = [base, base / "ehrshot_sequence_datasets", base / "sequence_datasets"]
    for c in candidates:
        if looks_like_sequence_root(c):
            return c
    if base.exists():
        for c in base.rglob("examples.parquet"):
            possible_root = c.parent.parent.parent
            if looks_like_sequence_root(possible_root):
                return possible_root
    raise FileNotFoundError(
        f"Could not find sequence dataset root under {base}. Expected: <root>/<task>/<version>/examples.parquet"
    )

if USE_CLEARML_DOWNLOAD:
    SEQUENCE_DATA_DIR = find_sequence_dataset_root(download_sequence_artifact_from_clearml())
elif USE_LOCAL_SEQUENCE_DATASETS:
    SEQUENCE_DATA_DIR = find_sequence_dataset_root(LOCAL_SEQUENCE_DATA_DIR)
else:
    raise ValueError("Set USE_LOCAL_SEQUENCE_DATASETS or USE_CLEARML_DOWNLOAD")

print("SEQUENCE_DATA_DIR:", SEQUENCE_DATA_DIR.resolve())

SEQUENCE_DATA_DIR: /Users/polinakorobeinikova/Практика/ehrshot_sequence_datasets


## 2. Optional ClearML tracking

In [4]:
ENABLE_CLEARML = False
CLEARML_PROJECT = "EHR_Risk_Profiling/EHRSHOT"
CLEARML_TASK_NAME = "sequence_numeric_time_notebook_v1"
CLEARML_OUTPUT_URI = "s3://api.blackhole2.ai.innopolis.university:443/pershin-medailab"

task = None
if ENABLE_CLEARML:
    from clearml import Task
    task = Task.init(
        project_name=CLEARML_PROJECT,
        task_name=CLEARML_TASK_NAME,
        output_uri=CLEARML_OUTPUT_URI,
    )
    task.connect({
        "run_tasks": RUN_TASKS,
        "run_sequence_versions": RUN_SEQUENCE_VERSIONS,
        "run_model_names": RUN_MODEL_NAMES,
        "max_len": MAX_LEN,
        "batch_size": BATCH_SIZE,
        "n_epochs": N_EPOCHS,
        "debug_run": DEBUG_RUN,
    })

## 3. Load data helpers

In [5]:
def load_sequence_examples(task_name: str, version: str) -> pd.DataFrame:
    path = SEQUENCE_DATA_DIR / task_name / version / "examples.parquet"
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_parquet(path)


def load_vocab(task_name: str, version: str) -> dict[str, int]:
    path = SEQUENCE_DATA_DIR / task_name / version / "vocab.json"
    if not path.exists():
        raise FileNotFoundError(path)
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def to_plain_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, np.ndarray):
        return x.tolist()
    if x is None:
        return []
    if isinstance(x, float) and math.isnan(x):
        return []
    return list(x) if hasattr(x, "__iter__") and not isinstance(x, str) else []

rows = []
for task_name in SELECTED_TASKS:
    for version in SEQUENCE_VERSIONS:
        p = SEQUENCE_DATA_DIR / task_name / version / "examples.parquet"
        rows.append({"task": task_name, "version": version, "exists": p.exists(), "path": str(p)})
available_df = pd.DataFrame(rows)
display(available_df)
assert available_df["exists"].all(), "Some sequence datasets are missing"

,task,version,exists,path
0,guo_readmission,raw,True,ehrshot_sequence_datasets/guo_readmission/raw/...
1,guo_readmission,compressed_dedup,True,ehrshot_sequence_datasets/guo_readmission/comp...
2,guo_readmission,compressed_first_last,True,ehrshot_sequence_datasets/guo_readmission/comp...
3,guo_readmission,compressed_condition_era,True,ehrshot_sequence_datasets/guo_readmission/comp...
4,guo_readmission,compressed_hybrid,True,ehrshot_sequence_datasets/guo_readmission/comp...
5,guo_icu,raw,True,ehrshot_sequence_datasets/guo_icu/raw/examples...
6,guo_icu,compressed_dedup,True,ehrshot_sequence_datasets/guo_icu/compressed_d...
7,guo_icu,compressed_first_last,True,ehrshot_sequence_datasets/guo_icu/compressed_f...
8,guo_icu,compressed_condition_era,True,ehrshot_sequence_datasets/guo_icu/compressed_c...
9,guo_icu,compressed_hybrid,True,ehrshot_sequence_datasets/guo_icu/compressed_h...


## 4. Numeric coverage and sequence length audit

In [6]:
def numeric_coverage_for_df(df: pd.DataFrame) -> dict:
    total_events = 0
    numeric_events = 0
    seq_lens = []
    for _, row in df.iterrows():
        nums = to_plain_list(row.get("numeric_values", []))
        toks = to_plain_list(row.get("token_ids", []))
        total_events += len(toks)
        seq_lens.append(len(toks))
        if len(nums) < len(toks):
            nums = nums + [np.nan] * (len(toks) - len(nums))
        for v in nums[:len(toks)]:
            try:
                if np.isfinite(float(v)):
                    numeric_events += 1
            except Exception:
                pass
    return {
        "n_examples": len(df),
        "total_events": total_events,
        "numeric_events": numeric_events,
        "numeric_event_share": numeric_events / total_events if total_events else np.nan,
        "mean_seq_len": float(np.mean(seq_lens)) if seq_lens else np.nan,
        "median_seq_len": float(np.median(seq_lens)) if seq_lens else np.nan,
        "p90_seq_len": float(np.quantile(seq_lens, 0.90)) if seq_lens else np.nan,
    }

coverage_rows = []
for task_name in SELECTED_TASKS:
    for version in SEQUENCE_VERSIONS:
        df = load_sequence_examples(task_name, version)
        for split, part in df.groupby("split"):
            row = numeric_coverage_for_df(part)
            row.update({"task": task_name, "version": version, "split": split, "event_rate": part["label"].mean()})
            coverage_rows.append(row)

numeric_coverage_df = pd.DataFrame(coverage_rows)
display(numeric_coverage_df.sort_values(["task", "version", "split"]))
numeric_coverage_df.to_csv(RESULTS_DIR / "numeric_coverage_sequence_length.csv", index=False)

,n_examples,total_events,numeric_events,numeric_event_share,mean_seq_len,median_seq_len,p90_seq_len,task,version,split,event_rate
24,2037,13136807,9238389,0.703245,6449.095238,3005.0,17397.4,guo_icu,compressed_condition_era,held_out,0.041728
25,2402,17505331,12243042,0.699389,7287.814738,3221.0,18927.0,guo_icu,compressed_condition_era,train,0.047044
26,2052,12345739,8776934,0.710928,6016.442008,2873.0,15211.3,guo_icu,compressed_condition_era,tuning,0.044834
18,2037,13243822,9227538,0.696743,6501.630830,3026.0,17489.4,guo_icu,compressed_dedup,held_out,0.041728
19,2402,17672592,12228257,0.691933,7357.448793,3261.5,18998.5,guo_icu,compressed_dedup,train,0.047044
20,2052,12452316,8765767,0.703947,6068.380117,2901.0,15416.9,guo_icu,compressed_dedup,tuning,0.044834
21,2037,13135868,9227538,0.702469,6448.634266,3005.0,17397.4,guo_icu,compressed_first_last,held_out,0.041728
22,2402,17503990,12228257,0.698598,7287.256453,3220.0,18926.9,guo_icu,compressed_first_last,train,0.047044
23,2052,12344778,8765767,0.710079,6015.973684,2873.0,15211.2,guo_icu,compressed_first_last,tuning,0.044834
27,2037,13135868,9238389,0.703295,6448.634266,3005.0,17397.4,guo_icu,compressed_hybrid,held_out,0.041728


## 5. Token-specific numeric normalization

`numeric_value` нельзя подавать raw без нормализации: разные коды живут в разных шкалах. Поэтому считаем mean/std по `token_id` на train split, а в модель подаем `[z_value, numeric_present]`.

In [7]:
def compute_token_numeric_stats(df_train: pd.DataFrame, vocab_size: int, min_count: int = 3):
    count = np.zeros(vocab_size, dtype=np.float64)
    sum_x = np.zeros(vocab_size, dtype=np.float64)
    sum_x2 = np.zeros(vocab_size, dtype=np.float64)

    for _, row in tqdm(df_train.iterrows(), total=len(df_train), desc="numeric stats"):
        tokens = to_plain_list(row["token_ids"])
        nums = to_plain_list(row.get("numeric_values", []))
        if len(nums) < len(tokens):
            nums = nums + [np.nan] * (len(tokens) - len(nums))
        for token_id, value in zip(tokens, nums):
            token_id = int(token_id)
            if token_id < 0 or token_id >= vocab_size:
                continue
            try:
                x = float(value)
            except Exception:
                continue
            if not np.isfinite(x):
                continue
            count[token_id] += 1
            sum_x[token_id] += x
            sum_x2[token_id] += x * x

    mean = np.zeros(vocab_size, dtype=np.float32)
    std = np.ones(vocab_size, dtype=np.float32)
    valid = count >= min_count
    mean[valid] = (sum_x[valid] / count[valid]).astype(np.float32)
    var = (sum_x2[valid] / count[valid]) - mean[valid] ** 2
    std[valid] = np.sqrt(np.maximum(var, 1e-6)).astype(np.float32)

    stats_df = pd.DataFrame({
        "token_id": np.arange(vocab_size),
        "numeric_count_train": count.astype(int),
        "numeric_mean_train": mean,
        "numeric_std_train": std,
        "has_enough_numeric_stats": valid,
    })
    return mean, std, stats_df

## 6. Dataset and collate

In [8]:
class EHRSequenceNumericDataset(Dataset):
    def __init__(self, df: pd.DataFrame, max_len: int = MAX_LEN):
        self.df = df.reset_index(drop=True)
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        token_ids = to_plain_list(row["token_ids"])
        days_before = to_plain_list(row["days_before_prediction"])
        delta_days = to_plain_list(row["delta_days"])
        numeric_values = to_plain_list(row.get("numeric_values", []))

        if len(token_ids) == 0:
            token_ids = [UNK_ID]
            days_before = [0.0]
            delta_days = [0.0]
            numeric_values = [np.nan]

        n = len(token_ids)
        if len(days_before) < n:
            days_before += [0.0] * (n - len(days_before))
        if len(delta_days) < n:
            delta_days += [0.0] * (n - len(delta_days))
        if len(numeric_values) < n:
            numeric_values += [np.nan] * (n - len(numeric_values))

        return {
            "row_id": int(row["row_id"]),
            "subject_id": int(row["subject_id"]),
            "tokens": token_ids[-self.max_len:],
            "days_before": days_before[-self.max_len:],
            "delta_days": delta_days[-self.max_len:],
            "numeric_values": numeric_values[-self.max_len:],
            "label": int(row["label"]),
        }


def make_numeric_collate_fn(token_numeric_mean: np.ndarray, token_numeric_std: np.ndarray):
    mean_t = torch.tensor(token_numeric_mean, dtype=torch.float32)
    std_t = torch.tensor(token_numeric_std, dtype=torch.float32)

    def collate(batch):
        batch_size = len(batch)
        lengths = torch.tensor([len(x["tokens"]) for x in batch], dtype=torch.long)
        max_len = int(lengths.max().item())

        tokens = torch.full((batch_size, max_len), PAD_ID, dtype=torch.long)
        time_features = torch.zeros((batch_size, max_len, 2), dtype=torch.float32)
        numeric_features = torch.zeros((batch_size, max_len, 2), dtype=torch.float32)
        mask = torch.zeros((batch_size, max_len), dtype=torch.bool)
        labels = torch.tensor([x["label"] for x in batch], dtype=torch.float32)
        row_ids = torch.tensor([x["row_id"] for x in batch], dtype=torch.long)
        subject_ids = torch.tensor([x["subject_id"] for x in batch], dtype=torch.long)

        for i, item in enumerate(batch):
            l = len(item["tokens"])
            tok = torch.tensor(item["tokens"], dtype=torch.long)
            tokens[i, :l] = tok
            mask[i, :l] = True

            days_before = torch.tensor(item["days_before"], dtype=torch.float32).clamp(min=0)
            delta_days = torch.tensor(item["delta_days"], dtype=torch.float32).clamp(min=0)
            time_features[i, :l, 0] = torch.log1p(days_before)
            time_features[i, :l, 1] = torch.log1p(delta_days)

            raw_nums, present = [], []
            for v in item["numeric_values"]:
                try:
                    x = float(v)
                    ok = np.isfinite(x)
                except Exception:
                    x, ok = 0.0, False
                raw_nums.append(x if ok else 0.0)
                present.append(1.0 if ok else 0.0)

            raw_nums = torch.tensor(raw_nums, dtype=torch.float32)
            present = torch.tensor(present, dtype=torch.float32)
            m = mean_t[tok]
            s = std_t[tok].clamp(min=1e-6)
            z = ((raw_nums - m) / s).clamp(min=-10.0, max=10.0)
            z = torch.where(present > 0, z, torch.zeros_like(z))

            numeric_features[i, :l, 0] = z
            numeric_features[i, :l, 1] = present

        return {
            "tokens": tokens,
            "time_features": time_features,
            "numeric_features": numeric_features,
            "mask": mask,
            "lengths": lengths,
            "labels": labels,
            "row_ids": row_ids,
            "subject_ids": subject_ids,
        }
    return collate


def maybe_subsample(df: pd.DataFrame, max_examples: int | None):
    if max_examples is None or len(df) <= max_examples:
        return df
    return df.sample(n=max_examples, random_state=RANDOM_STATE).reset_index(drop=True)


def make_loaders(df: pd.DataFrame, token_numeric_mean: np.ndarray, token_numeric_std: np.ndarray):
    split_dfs = {}
    for split in ["train", "tuning", "held_out"]:
        part = df[df["split"] == split].copy().reset_index(drop=True)
        part = maybe_subsample(part, MAX_TRAIN_EXAMPLES if split == "train" else MAX_EVAL_EXAMPLES)
        split_dfs[split] = part

    collate_fn = make_numeric_collate_fn(token_numeric_mean, token_numeric_std)
    loaders = {
        split: DataLoader(
            EHRSequenceNumericDataset(part, max_len=MAX_LEN),
            batch_size=BATCH_SIZE,
            shuffle=(split == "train"),
            num_workers=NUM_WORKERS,
            collate_fn=collate_fn,
        )
        for split, part in split_dfs.items()
    }
    return loaders, split_dfs

## 7. Model components: numeric embedding + Time2Vec

In [9]:
class Time2VecEncoder(nn.Module):
    def __init__(self, input_dim: int = 2, out_dim: int = 64):
        super().__init__()
        linear_dim = max(1, out_dim // 2)
        periodic_dim = out_dim - linear_dim
        self.linear = nn.Linear(input_dim, linear_dim)
        self.periodic = nn.Linear(input_dim, periodic_dim) if periodic_dim > 0 else None

    def forward(self, x):
        parts = [self.linear(x)]
        if self.periodic is not None:
            parts.append(torch.sin(self.periodic(x)))
        return torch.cat(parts, dim=-1)


class EventEmbeddingNumeric(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int = 64, dropout: float = 0.20):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_ID)
        self.time_proj = Time2VecEncoder(input_dim=2, out_dim=emb_dim)
        self.numeric_proj = nn.Sequential(nn.Linear(2, emb_dim), nn.GELU(), nn.LayerNorm(emb_dim))
        self.norm = nn.LayerNorm(emb_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, tokens, time_features, numeric_features):
        x = self.token_emb(tokens) + self.time_proj(time_features) + self.numeric_proj(numeric_features)
        return self.dropout(self.norm(x))


def reverse_by_lengths(x, lengths):
    out = x.clone()
    for i, l in enumerate(lengths.tolist()):
        out[i, :l] = torch.flip(x[i, :l], dims=[0])
    return out

In [10]:
class RNNNumericClassifier(nn.Module):
    def __init__(self, vocab_size: int, rnn_type: str = "GRU", emb_dim: int = 64, hidden_dim: int = 128, num_layers: int = 1, dropout: float = 0.20):
        super().__init__()
        self.model_config = {"model_class": "RNNNumericClassifier", "rnn_type": rnn_type, "emb_dim": emb_dim, "hidden_dim": hidden_dim, "num_layers": num_layers, "dropout": dropout}
        self.event_emb = EventEmbeddingNumeric(vocab_size, emb_dim, dropout=dropout)
        rnn_cls = nn.GRU if rnn_type == "GRU" else nn.LSTM
        self.rnn = rnn_cls(emb_dim, hidden_dim, num_layers=num_layers, dropout=dropout if num_layers > 1 else 0.0, batch_first=True)
        self.head = nn.Sequential(nn.LayerNorm(hidden_dim), nn.Dropout(dropout), nn.Linear(hidden_dim, 1))

    def forward(self, tokens, time_features, numeric_features, mask, lengths):
        x = self.event_emb(tokens, time_features, numeric_features)
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h = self.rnn(packed)
        if isinstance(h, tuple):
            h = h[0]
        return self.head(h[-1]).squeeze(-1)


class RetainLiteNumericClassifier(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int = 64, hidden_dim: int = 128, dropout: float = 0.20):
        super().__init__()
        self.model_config = {"model_class": "RetainLiteNumericClassifier", "emb_dim": emb_dim, "hidden_dim": hidden_dim, "dropout": dropout}
        self.event_emb = EventEmbeddingNumeric(vocab_size, emb_dim, dropout=dropout)
        self.alpha_gru = nn.GRU(emb_dim, hidden_dim, batch_first=True)
        self.beta_gru = nn.GRU(emb_dim, hidden_dim, batch_first=True)
        self.alpha_fc = nn.Linear(hidden_dim, 1)
        self.beta_fc = nn.Linear(hidden_dim, emb_dim)
        self.head = nn.Sequential(nn.LayerNorm(emb_dim), nn.Dropout(dropout), nn.Linear(emb_dim, 1))

    def forward(self, tokens, time_features, numeric_features, mask, lengths):
        x = self.event_emb(tokens, time_features, numeric_features)
        x_rev = reverse_by_lengths(x, lengths)
        mask_rev = reverse_by_lengths(mask.unsqueeze(-1).float(), lengths).squeeze(-1).bool()
        alpha_h, _ = self.alpha_gru(x_rev)
        beta_h, _ = self.beta_gru(x_rev)
        alpha_logits = self.alpha_fc(alpha_h).squeeze(-1).masked_fill(~mask_rev, -1e9)
        alpha = torch.softmax(alpha_logits, dim=1)
        beta = torch.tanh(self.beta_fc(beta_h))
        context = torch.sum(alpha.unsqueeze(-1) * beta * x_rev, dim=1)
        return self.head(context).squeeze(-1)

In [11]:
def make_model_zoo(vocab_size: int) -> dict[str, nn.Module]:
    return {
        "GRU_1L_numeric": RNNNumericClassifier(vocab_size, rnn_type="GRU", num_layers=1),
        "GRU_2L_numeric": RNNNumericClassifier(vocab_size, rnn_type="GRU", num_layers=2),
        "LSTM_1L_numeric": RNNNumericClassifier(vocab_size, rnn_type="LSTM", num_layers=1),
        "LSTM_2L_numeric": RNNNumericClassifier(vocab_size, rnn_type="LSTM", num_layers=2),
        "RETAIN_lite_numeric": RetainLiteNumericClassifier(vocab_size),
    }

## 8. Training and evaluation

In [12]:
def sigmoid_np(x):
    x = np.asarray(x, dtype=float)
    return 1 / (1 + np.exp(-x))


def compute_ranking_metrics(y_true, y_prob) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    return {
        "auroc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan,
        "auprc": average_precision_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan,
        "brier": brier_score_loss(y_true, y_prob),
        "logloss": log_loss(y_true, np.clip(y_prob, 1e-6, 1 - 1e-6), labels=[0, 1]),
        "event_rate": float(y_true.mean()),
        "n": int(len(y_true)),
        "n_positive": int(y_true.sum()),
    }


def compute_topk_metrics(y_true, y_prob, top_fracs=(0.01, 0.05, 0.10, 0.20)) -> pd.DataFrame:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    n = len(y_true)
    n_pos = int(y_true.sum())
    base_rate = float(y_true.mean()) if n else np.nan
    order = np.argsort(-y_prob)
    rows = []
    for frac in top_fracs:
        k = max(1, int(np.ceil(n * frac)))
        idx = order[:k]
        selected_events = int(y_true[idx].sum())
        event_rate = selected_events / k
        rows.append({
            "top_frac": frac,
            "top_k": k,
            "events_in_top_k": selected_events,
            "top_k_event_rate": event_rate,
            "top_k_lift": event_rate / base_rate if base_rate > 0 else np.nan,
            "event_capture": selected_events / n_pos if n_pos > 0 else np.nan,
            "base_event_rate": base_rate,
            "n": n,
            "n_positive": n_pos,
        })
    return pd.DataFrame(rows)


def run_inference(model, loader):
    model.eval()
    all_logits, all_y, all_row_ids, all_subject_ids = [], [], [], []
    with torch.no_grad():
        for batch in loader:
            tokens = batch["tokens"].to(DEVICE)
            time_features = batch["time_features"].to(DEVICE)
            numeric_features = batch["numeric_features"].to(DEVICE)
            mask = batch["mask"].to(DEVICE)
            lengths = batch["lengths"].to(DEVICE)
            logits = model(tokens, time_features, numeric_features, mask, lengths)
            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(batch["labels"].numpy())
            all_row_ids.append(batch["row_ids"].numpy())
            all_subject_ids.append(batch["subject_ids"].numpy())
    return {
        "logits": np.concatenate(all_logits),
        "y": np.concatenate(all_y).astype(int),
        "row_id": np.concatenate(all_row_ids).astype(int),
        "subject_id": np.concatenate(all_subject_ids).astype(int),
    }

In [13]:
def train_sequence_model(model, loaders):
    model = model.to(DEVICE)
    train_labels = []
    for batch in loaders["train"]:
        train_labels.append(batch["labels"].numpy())
    train_labels = np.concatenate(train_labels)
    n_pos = int(train_labels.sum())
    n_neg = int(len(train_labels) - n_pos)
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32, device=DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    history = []
    best_state = None
    best_tuning_auprc = -np.inf
    bad_epochs = 0

    for epoch in range(1, N_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        n_batches = 0
        for batch in tqdm(loaders["train"], desc=f"epoch {epoch}"):
            optimizer.zero_grad()
            tokens = batch["tokens"].to(DEVICE)
            time_features = batch["time_features"].to(DEVICE)
            numeric_features = batch["numeric_features"].to(DEVICE)
            mask = batch["mask"].to(DEVICE)
            lengths = batch["lengths"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            logits = model(tokens, time_features, numeric_features, mask, lengths)
            loss = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            total_loss += float(loss.detach().cpu().item())
            n_batches += 1

        tuning_pred = run_inference(model, loaders["tuning"])
        tuning_prob = sigmoid_np(tuning_pred["logits"])
        tuning_metrics = compute_ranking_metrics(tuning_pred["y"], tuning_prob)
        row = {
            "epoch": epoch,
            "train_loss": total_loss / max(n_batches, 1),
            "tuning_auroc": tuning_metrics["auroc"],
            "tuning_auprc": tuning_metrics["auprc"],
            "tuning_brier": tuning_metrics["brier"],
            "tuning_logloss": tuning_metrics["logloss"],
        }
        history.append(row)
        print(row)

        if tuning_metrics["auprc"] > best_tuning_auprc:
            best_tuning_auprc = tuning_metrics["auprc"]
            best_state = deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print(f"Early stopping at epoch {epoch}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

In [14]:
def evaluate_model_with_calibration(task_name, version, model_name, model, loaders):
    tuning = run_inference(model, loaders["tuning"])
    heldout = run_inference(model, loaders["held_out"])
    rows, pred_tables, topk_tables = [], [], []

    probs = {"raw": sigmoid_np(heldout["logits"])}
    try:
        calibrator = LogisticRegression(solver="lbfgs")
        calibrator.fit(tuning["logits"].reshape(-1, 1), tuning["y"])
        probs["platt"] = calibrator.predict_proba(heldout["logits"].reshape(-1, 1))[:, 1]
    except Exception as e:
        print("Platt calibration failed:", repr(e))

    for calibration, held_prob in probs.items():
        m = compute_ranking_metrics(heldout["y"], held_prob)
        rows.append({
            "task": task_name,
            "version": version,
            "model": model_name,
            "calibration": calibration,
            "n_tuning": int(len(tuning["y"])),
            "n_heldout": int(len(heldout["y"])),
            "n_positive_heldout": int(heldout["y"].sum()),
            "event_rate_heldout": float(heldout["y"].mean()),
            "heldout_auroc": m["auroc"],
            "heldout_auprc": m["auprc"],
            "heldout_brier": m["brier"],
            "heldout_logloss": m["logloss"],
        })
        pred_tables.append(pd.DataFrame({
            "task": task_name,
            "version": version,
            "model": model_name,
            "calibration": calibration,
            "row_id": heldout["row_id"],
            "subject_id": heldout["subject_id"],
            "y_true": heldout["y"],
            "risk": held_prob,
            "logit": heldout["logits"],
        }))
        tk = compute_topk_metrics(heldout["y"], held_prob)
        tk.insert(0, "calibration", calibration)
        tk.insert(0, "model", model_name)
        tk.insert(0, "version", version)
        tk.insert(0, "task", task_name)
        topk_tables.append(tk)

    return pd.DataFrame(rows), pd.concat(pred_tables, ignore_index=True), pd.concat(topk_tables, ignore_index=True)

## 9. Run experiments

In [15]:
def run_one_experiment(task_name: str, version: str, model_name: str):
    print("\n" + "=" * 100)
    print(f"Experiment: {task_name} | {version} | {model_name}")

    df = load_sequence_examples(task_name, version)
    vocab = load_vocab(task_name, version)
    vocab_size = len(vocab)

    df_train_full = df[df["split"] == "train"].copy().reset_index(drop=True)
    token_mean, token_std, token_stats_df = compute_token_numeric_stats(df_train_full, vocab_size=vocab_size)

    loaders, split_dfs = make_loaders(df, token_mean, token_std)
    model = make_model_zoo(vocab_size)[model_name]

    model, history_df = train_sequence_model(model, loaders)
    result_df, pred_df, topk_df = evaluate_model_with_calibration(task_name, version, model_name, model, loaders)

    exp_name = f"{task_name}__{version}__{model_name}"
    history_df.to_csv(RESULTS_DIR / f"{exp_name}__history.csv", index=False)
    pred_df.to_csv(RESULTS_DIR / f"{exp_name}__heldout_predictions.csv", index=False)
    topk_df.to_csv(RESULTS_DIR / f"{exp_name}__topk.csv", index=False)
    token_stats_df.to_csv(RESULTS_DIR / f"{exp_name}__token_numeric_stats.csv", index=False)

    if SAVE_MODEL_CHECKPOINTS:
        ckpt_path = CHECKPOINT_DIR / f"{exp_name}.pt"
        torch.save({
            "task": task_name,
            "version": version,
            "model_name": model_name,
            "vocab_size": vocab_size,
            "max_len": MAX_LEN,
            "model_config": getattr(model, "model_config", {}),
            "token_numeric_mean": token_mean,
            "token_numeric_std": token_std,
            "state_dict": model.state_dict(),
        }, ckpt_path)

    display(result_df[["task", "version", "model", "calibration", "heldout_auroc", "heldout_auprc", "heldout_brier", "heldout_logloss"]])
    display(topk_df[topk_df["calibration"] == "platt"] if "platt" in topk_df["calibration"].values else topk_df)
    return result_df, pred_df, topk_df, history_df

all_result_tables, all_prediction_tables, all_topk_tables, all_history_tables = [], [], [], []

for task_name in RUN_TASKS:
    for version in RUN_SEQUENCE_VERSIONS:
        for model_name in RUN_MODEL_NAMES:
            result_df, pred_df, topk_df, history_df = run_one_experiment(task_name, version, model_name)
            all_result_tables.append(result_df)
            all_prediction_tables.append(pred_df)
            all_topk_tables.append(topk_df)
            history_df = history_df.copy()
            history_df["task"] = task_name
            history_df["version"] = version
            history_df["model"] = model_name
            all_history_tables.append(history_df)

sequence_numeric_results_df = pd.concat(all_result_tables, ignore_index=True)
sequence_numeric_predictions_df = pd.concat(all_prediction_tables, ignore_index=True)
sequence_numeric_topk_df = pd.concat(all_topk_tables, ignore_index=True)
sequence_numeric_history_df = pd.concat(all_history_tables, ignore_index=True)

sequence_numeric_results_df.to_csv(RESULTS_DIR / "sequence_numeric_results.csv", index=False)
sequence_numeric_predictions_df.to_csv(RESULTS_DIR / "sequence_numeric_heldout_predictions.csv", index=False)
sequence_numeric_topk_df.to_csv(RESULTS_DIR / "sequence_numeric_topk_risk_stratification.csv", index=False)
sequence_numeric_history_df.to_csv(RESULTS_DIR / "sequence_numeric_training_history.csv", index=False)

display(sequence_numeric_results_df.sort_values(["task", "heldout_auprc"], ascending=[True, False]))


Experiment: guo_readmission | raw | GRU_1L_numeric


numeric stats:   0%|          | 0/2608 [00:00<?, ?it/s]

epoch 1:   0%|          | 0/82 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 1.227457293649999, 'tuning_auroc': np.float64(0.6571613439940842), 'tuning_auprc': np.float64(0.24259363702284736), 'tuning_brier': np.float64(0.31054664393093645), 'tuning_logloss': 0.8183455709529407}


epoch 2:   0%|          | 0/82 [00:00<?, ?it/s]

...

epoch 8:   0%|          | 0/76 [00:00<?, ?it/s]

{'epoch': 8, 'train_loss': 1.0773397035112506, 'tuning_auroc': np.float64(0.7645629991126885), 'tuning_auprc': np.float64(0.22935361384996175), 'tuning_brier': np.float64(0.10138038260525525), 'tuning_logloss': 0.3518124635834262}
Early stopping at epoch 8


,task,version,model,calibration,heldout_auroc,heldout_auprc,heldout_brier,heldout_logloss
0,guo_icu,compressed_hybrid,RETAIN_lite_numeric,raw,0.731527,0.164009,0.065393,0.266272
1,guo_icu,compressed_hybrid,RETAIN_lite_numeric,platt,0.731527,0.164009,0.037441,0.154758


,task,version,model,calibration,top_frac,top_k,events_in_top_k,top_k_event_rate,top_k_lift,event_capture,base_event_rate,n,n_positive
4,guo_icu,compressed_hybrid,RETAIN_lite_numeric,platt,0.01,21,7,0.333333,7.988235,0.082353,0.041728,2037,85
5,guo_icu,compressed_hybrid,RETAIN_lite_numeric,platt,0.05,102,24,0.235294,5.638754,0.282353,0.041728,2037,85
6,guo_icu,compressed_hybrid,RETAIN_lite_numeric,platt,0.10,204,35,0.171569,4.111592,0.411765,0.041728,2037,85
7,guo_icu,compressed_hybrid,RETAIN_lite_numeric,platt,0.20,408,46,0.112745,2.701903,0.541176,0.041728,2037,85


,task,version,model,calibration,n_tuning,n_heldout,n_positive_heldout,event_rate_heldout,heldout_auroc,heldout_auprc,heldout_brier,heldout_logloss
92,guo_icu,compressed_hybrid,GRU_2L_numeric,raw,2052,2037,85,0.041728,0.793605,0.215872,0.056325,0.236349
93,guo_icu,compressed_hybrid,GRU_2L_numeric,platt,2052,2037,85,0.041728,0.793605,0.215872,0.038011,0.158068
50,guo_icu,raw,GRU_1L_numeric,raw,2052,2037,85,0.041728,0.724602,0.180168,0.079329,0.289808
51,guo_icu,raw,GRU_1L_numeric,platt,2052,2037,85,0.041728,0.724602,0.180168,0.037878,0.156879
70,guo_icu,compressed_first_last,GRU_1L_numeric,raw,2052,2037,85,0.041728,0.713410,0.177232,0.042135,0.169255
...,...,...,...,...,...,...,...,...,...,...,...,...
1,guo_readmission,raw,GRU_1L_numeric,platt,2206,2189,260,0.118776,0.687341,0.267068,0.097802,0.338444
14,guo_readmission,compressed_dedup,LSTM_1L_numeric,raw,2206,2189,260,0.118776,0.703348,0.261853,0.165523,0.515394
15,guo_readmission,compressed_dedup,LSTM_1L_numeric,platt,2206,2189,260,0.118776,0.703348,0.261853,0.097076,0.334172
30,guo_readmission,compressed_condition_era,GRU_1L_numeric,raw,2206,2189,260,0.118776,0.684332,0.260262,0.230141,0.658154


## 10. Compare with previous code-only sequence baseline

In [19]:
PREVIOUS_SEQUENCE_RESULTS_PATH = Path("ehrshot_sequence_results/sequence_baseline_results.csv")

if PREVIOUS_SEQUENCE_RESULTS_PATH.exists():
    prev = pd.read_csv(PREVIOUS_SEQUENCE_RESULTS_PATH)
    prev = prev[prev["calibration"] == "platt"].copy()
    curr = sequence_numeric_results_df[sequence_numeric_results_df["calibration"] == "platt"].copy()

    prev_best = (
        prev.sort_values(["task", "version", "heldout_auprc"], ascending=[True, True, False])
        .groupby(["task", "version"], as_index=False)
        .head(1)
        [["task", "version", "model", "heldout_auroc", "heldout_auprc", "heldout_brier", "heldout_logloss"]]
        .rename(columns={"model": "best_code_only_model", "heldout_auroc": "code_only_auroc", "heldout_auprc": "code_only_auprc", "heldout_brier": "code_only_brier", "heldout_logloss": "code_only_logloss"})
    )
    curr_best = (
        curr.sort_values(["task", "version", "heldout_auprc"], ascending=[True, True, False])
        .groupby(["task", "version"], as_index=False)
        .head(1)
        [["task", "version", "model", "heldout_auroc", "heldout_auprc", "heldout_brier", "heldout_logloss"]]
        .rename(columns={"model": "best_numeric_model", "heldout_auroc": "numeric_auroc", "heldout_auprc": "numeric_auprc", "heldout_brier": "numeric_brier", "heldout_logloss": "numeric_logloss"})
    )
    cmp_df = prev_best.merge(curr_best, on=["task", "version"], how="inner")
    cmp_df["delta_auroc_numeric_minus_code_only"] = cmp_df["numeric_auroc"] - cmp_df["code_only_auroc"]
    cmp_df["delta_auprc_numeric_minus_code_only"] = cmp_df["numeric_auprc"] - cmp_df["code_only_auprc"]
    cmp_df["delta_brier_numeric_minus_code_only"] = cmp_df["numeric_brier"] - cmp_df["code_only_brier"]
    cmp_df["delta_logloss_numeric_minus_code_only"] = cmp_df["numeric_logloss"] - cmp_df["code_only_logloss"]
    cmp_df.to_csv(RESULTS_DIR / "numeric_vs_code_only_best_by_task_version.csv", index=False)
    display(cmp_df)
else:
    print("Previous results CSV not found:", PREVIOUS_SEQUENCE_RESULTS_PATH)

,task,version,best_code_only_model,code_only_auroc,code_only_auprc,code_only_brier,code_only_logloss,best_numeric_model,numeric_auroc,numeric_auprc,numeric_brier,numeric_logloss,delta_auroc_numeric_minus_code_only,delta_auprc_numeric_minus_code_only,delta_brier_numeric_minus_code_only,delta_logloss_numeric_minus_code_only
0,guo_icu,compressed_condition_era,LSTM_1L,0.722191,0.184673,0.037907,0.157627,GRU_2L_numeric,0.770287,0.170596,0.038209,0.158038,0.048095,-0.014077,0.000302,0.000410
1,guo_icu,compressed_dedup,GRU_1L,0.745474,0.151499,0.037885,0.155611,RETAIN_lite_numeric,0.753236,0.158373,0.038032,0.155685,0.007763,0.006874,0.000147,0.000074
2,guo_icu,compressed_first_last,GRU_2L,0.715929,0.149425,0.038502,0.160899,GRU_1L_numeric,0.713410,0.177232,0.037622,0.155315,-0.002519,0.027806,-0.000881,-0.005584
3,guo_icu,compressed_hybrid,LSTM_2L,0.693846,0.152087,0.038475,0.162373,GRU_2L_numeric,0.793605,0.215872,0.038011,0.158068,0.099759,0.063785,-0.000463,-0.004306
4,guo_icu,raw,LSTM_1L,0.711391,0.146446,0.038049,0.157459,GRU_1L_numeric,0.724602,0.180168,0.037878,0.156879,0.013211,0.033722,-0.000170,-0.000580
5,guo_readmission,compressed_condition_era,RETAIN_lite,0.769033,0.338576,0.092244,0.313725,RETAIN_lite_numeric,0.754528,0.345361,0.091972,0.316718,-0.014505,0.006785,-0.000272,0.002993
6,guo_readmission,compressed_dedup,RETAIN_lite,0.781585,0.382325,0.089831,0.307238,RETAIN_lite_numeric,0.776034,0.376194,0.088144,0.304184,-0.005551,-0.006132,-0.001687,-0.003055
7,guo_readmission,compressed_first_last,RETAIN_lite,0.762739,0.353660,0.091631,0.314384,RETAIN_lite_numeric,0.771829,0.371733,0.089774,0.308885,0.009090,0.018073,-0.001856,-0.005499
8,guo_readmission,compressed_hybrid,RETAIN_lite,0.767205,0.364681,0.090536,0.310999,RETAIN_lite_numeric,0.777451,0.364969,0.091712,0.313266,0.010246,0.000288,0.001176,0.002266
9,guo_readmission,raw,RETAIN_lite,0.729218,0.314486,0.094349,0.325476,RETAIN_lite_numeric,0.772471,0.380255,0.089038,0.307085,0.043253,0.065768,-0.005311,-0.018391


## Bootstrap CI for numeric-aware sequence results

Bootstrap CI below is computed over held-out prediction examples. This is consistent with the previous code-only sequence baseline. A stricter patient-level bootstrap is possible, but was not used here because it is substantially slower and not required for the current comparison.

In [21]:
# Pick numeric results / predictions dataframes

if "sequence_numeric_results_df" in globals():
    numeric_results_df = sequence_numeric_results_df.copy()
else:
    raise NameError(
        "Не найден results dataframe. Ожидаю sequence_numeric_results_df или sequence_results_df."
    )


if "sequence_numeric_predictions_df" in globals():
    numeric_predictions_df = sequence_numeric_predictions_df.copy()
else:
    prediction_files = sorted(RESULTS_DIR.glob("*__heldout_predictions.csv"))

    if len(prediction_files) == 0:
        raise FileNotFoundError(
            f"No heldout prediction files found in {RESULTS_DIR}."
        )

    numeric_predictions_df = pd.concat(
        [pd.read_csv(path) for path in prediction_files],
        ignore_index=True,
    )


required_result_cols = {
    "task",
    "version",
    "model",
    "calibration",
    "heldout_auprc",
    "heldout_auroc",
    "heldout_brier",
}

missing_result_cols = required_result_cols - set(numeric_results_df.columns)
if missing_result_cols:
    raise ValueError(f"Missing result columns: {missing_result_cols}")


required_pred_cols = {
    "task",
    "version",
    "model",
    "calibration",
    "y_true",
    "risk",
}

missing_pred_cols = required_pred_cols - set(numeric_predictions_df.columns)
if missing_pred_cols:
    raise ValueError(f"Missing prediction columns: {missing_pred_cols}")


numeric_predictions_df["y_true"] = numeric_predictions_df["y_true"].astype(int)
numeric_predictions_df["risk"] = numeric_predictions_df["risk"].astype(float)

print("numeric_results_df:", numeric_results_df.shape)
print("numeric_predictions_df:", numeric_predictions_df.shape)

display(numeric_results_df.head())
display(numeric_predictions_df.head())

numeric_results_df: (100, 12)
numeric_predictions_df: (211300, 9)


,task,version,model,calibration,n_tuning,n_heldout,n_positive_heldout,event_rate_heldout,heldout_auroc,heldout_auprc,heldout_brier,heldout_logloss
0,guo_readmission,raw,GRU_1L_numeric,raw,2206,2189,260,0.118776,0.687341,0.267068,0.170771,0.519469
1,guo_readmission,raw,GRU_1L_numeric,platt,2206,2189,260,0.118776,0.687341,0.267068,0.097802,0.338444
2,guo_readmission,raw,GRU_2L_numeric,raw,2206,2189,260,0.118776,0.744148,0.328227,0.107992,0.358488
3,guo_readmission,raw,GRU_2L_numeric,platt,2206,2189,260,0.118776,0.744148,0.328227,0.092704,0.318963
4,guo_readmission,raw,LSTM_1L_numeric,raw,2206,2189,260,0.118776,0.735385,0.291786,0.189810,0.571372


,task,version,model,calibration,row_id,subject_id,y_true,risk,logit
0,guo_readmission,raw,GRU_1L_numeric,raw,3,115972225,0,0.204805,-1.356528
1,guo_readmission,raw,GRU_1L_numeric,raw,6,115973293,0,0.120283,-1.989750
2,guo_readmission,raw,GRU_1L_numeric,raw,7,115973293,0,0.177194,-1.535477
3,guo_readmission,raw,GRU_1L_numeric,raw,22,115972475,0,0.165649,-1.616783
4,guo_readmission,raw,GRU_1L_numeric,raw,23,115972475,0,0.246309,-1.118397


In [23]:
# Fast bootstrap CI for numeric-aware sequence results

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)

N_BOOTSTRAP = 1000
BOOTSTRAP_SEED = 42

BOOTSTRAP_CI_PATH = RESULTS_DIR / "sequence_numeric_bootstrap_ci.csv"


def bootstrap_metric_ci(
    y_true,
    y_prob,
    metric_fn,
    n_bootstrap: int = N_BOOTSTRAP,
    ci: float = 0.95,
    random_state: int = BOOTSTRAP_SEED,
):
    rng = np.random.default_rng(random_state)

    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    n = len(y_true)
    values = []

    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)

        y_b = y_true[idx]
        p_b = y_prob[idx]

        try:
            value = metric_fn(y_b, p_b)
        except Exception:
            value = np.nan

        values.append(value)

    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]

    alpha = 1 - ci

    return {
        "mean": float(np.mean(values)) if len(values) else np.nan,
        "ci_low": float(np.quantile(values, alpha / 2)) if len(values) else np.nan,
        "ci_high": float(np.quantile(values, 1 - alpha / 2)) if len(values) else np.nan,
        "n_bootstrap_valid": int(len(values)),
    }


def auroc_metric(y, p):
    return roc_auc_score(y, p) if len(np.unique(y)) > 1 else np.nan


def auprc_metric(y, p):
    return average_precision_score(y, p) if len(np.unique(y)) > 1 else np.nan


def brier_metric(y, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return brier_score_loss(y, p)


def logloss_metric(y, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return log_loss(y, p, labels=[0, 1])


def topk_precision_metric(k_fraction: float):
    def metric(y, p):
        y = np.asarray(y).astype(int)
        p = np.asarray(p).astype(float)

        n_top = max(1, int(np.ceil(len(y) * k_fraction)))
        order = np.argsort(-p)

        return float(y[order[:n_top]].mean())

    return metric

In [24]:
# Fast bootstrap for best numeric model per task/version

best_platt_numeric = (
    numeric_results_df[numeric_results_df["calibration"] == "platt"]
    .sort_values(
        ["task", "version", "heldout_auprc"],
        ascending=[True, True, False],
    )
    .groupby(["task", "version"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

display(
    best_platt_numeric[
        [
            "task",
            "version",
            "model",
            "heldout_auprc",
            "heldout_auroc",
            "heldout_brier",
        ]
    ]
)

bootstrap_rows = []

metrics_to_bootstrap = [
    ("auroc", auroc_metric),
    ("auprc", auprc_metric),
    ("brier", brier_metric),
    ("logloss", logloss_metric),
    ("top_1pct_precision", topk_precision_metric(0.01)),
    ("top_5pct_precision", topk_precision_metric(0.05)),
    ("top_10pct_precision", topk_precision_metric(0.10)),
    ("top_20pct_precision", topk_precision_metric(0.20)),
]

for _, row in best_platt_numeric.iterrows():
    pred = numeric_predictions_df[
        (numeric_predictions_df["task"] == row["task"])
        & (numeric_predictions_df["version"] == row["version"])
        & (numeric_predictions_df["model"] == row["model"])
        & (numeric_predictions_df["calibration"] == "platt")
    ].copy()

    if len(pred) == 0:
        print("No predictions for:", row["task"], row["version"], row["model"])
        continue

    y = pred["y_true"].to_numpy()
    p = pred["risk"].to_numpy()

    n = len(y)
    n_positive = int(y.sum())
    event_rate = float(y.mean())

    print(
        f'Bootstrap: task={row["task"]}, version={row["version"]}, '
        f'model={row["model"]}, n={n}, positives={n_positive}'
    )

    for metric_name, metric_fn in metrics_to_bootstrap:
        ci_result = bootstrap_metric_ci(
            y_true=y,
            y_prob=p,
            metric_fn=metric_fn,
            n_bootstrap=N_BOOTSTRAP,
            random_state=BOOTSTRAP_SEED,
        )

        bootstrap_rows.append({
            "task": row["task"],
            "version": row["version"],
            "model": row["model"],
            "calibration": "platt",
            "metric": metric_name,
            **ci_result,
            "n": n,
            "n_positive": n_positive,
            "event_rate": event_rate,
            "n_bootstrap_requested": N_BOOTSTRAP,
            "bootstrap_level": "prediction_example",
        })

sequence_numeric_bootstrap_ci_df = pd.DataFrame(bootstrap_rows)

sequence_numeric_bootstrap_ci_df.to_csv(BOOTSTRAP_CI_PATH, index=False)

print("Saved:", BOOTSTRAP_CI_PATH)

display(
    sequence_numeric_bootstrap_ci_df
    .sort_values(["task", "version", "metric"])
)

,task,version,model,heldout_auprc,heldout_auroc,heldout_brier
0,guo_icu,compressed_condition_era,GRU_2L_numeric,0.170596,0.770287,0.038209
1,guo_icu,compressed_dedup,RETAIN_lite_numeric,0.158373,0.753236,0.038032
2,guo_icu,compressed_first_last,GRU_1L_numeric,0.177232,0.713410,0.037622
3,guo_icu,compressed_hybrid,GRU_2L_numeric,0.215872,0.793605,0.038011
4,guo_icu,raw,GRU_1L_numeric,0.180168,0.724602,0.037878
5,guo_readmission,compressed_condition_era,RETAIN_lite_numeric,0.345361,0.754528,0.091972
6,guo_readmission,compressed_dedup,RETAIN_lite_numeric,0.376194,0.776034,0.088144
7,guo_readmission,compressed_first_last,RETAIN_lite_numeric,0.371733,0.771829,0.089774
8,guo_readmission,compressed_hybrid,RETAIN_lite_numeric,0.364969,0.777451,0.091712
9,guo_readmission,raw,RETAIN_lite_numeric,0.380255,0.772471,0.089038


Bootstrap: task=guo_icu, version=compressed_condition_era, model=GRU_2L_numeric, n=2037, positives=85
Bootstrap: task=guo_icu, version=compressed_dedup, model=RETAIN_lite_numeric, n=2037, positives=85
Bootstrap: task=guo_icu, version=compressed_first_last, model=GRU_1L_numeric, n=2037, positives=85
Bootstrap: task=guo_icu, version=compressed_hybrid, model=GRU_2L_numeric, n=2037, positives=85
Bootstrap: task=guo_icu, version=raw, model=GRU_1L_numeric, n=2037, positives=85
Bootstrap: task=guo_readmission, version=compressed_condition_era, model=RETAIN_lite_numeric, n=2189, positives=260
Bootstrap: task=guo_readmission, version=compressed_dedup, model=RETAIN_lite_numeric, n=2189, positives=260
Bootstrap: task=guo_readmission, version=compressed_first_last, model=RETAIN_lite_numeric, n=2189, positives=260
Bootstrap: task=guo_readmission, version=compressed_hybrid, model=RETAIN_lite_numeric, n=2189, positives=260
Bootstrap: task=guo_readmission, version=raw, model=RETAIN_lite_numeric, n=218

,task,version,model,calibration,metric,mean,ci_low,ci_high,n_bootstrap_valid,n,n_positive,event_rate,n_bootstrap_requested,bootstrap_level
1,guo_icu,compressed_condition_era,GRU_2L_numeric,platt,auprc,0.180442,0.113818,0.260746,1000,2037,85,0.041728,1000,prediction_example
0,guo_icu,compressed_condition_era,GRU_2L_numeric,platt,auroc,0.769994,0.720161,0.819707,1000,2037,85,0.041728,1000,prediction_example
2,guo_icu,compressed_condition_era,GRU_2L_numeric,platt,brier,0.038251,0.030879,0.045172,1000,2037,85,0.041728,1000,prediction_example
3,guo_icu,compressed_condition_era,GRU_2L_numeric,platt,logloss,0.158170,0.132536,0.182575,1000,2037,85,0.041728,1000,prediction_example
6,guo_icu,compressed_condition_era,GRU_2L_numeric,platt,top_10pct_precision,0.132833,0.088235,0.181373,1000,2037,85,0.041728,1000,prediction_example
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,guo_readmission,raw,RETAIN_lite_numeric,platt,logloss,0.307485,0.284928,0.332358,1000,2189,260,0.118776,1000,prediction_example
78,guo_readmission,raw,RETAIN_lite_numeric,platt,top_10pct_precision,0.430566,0.369863,0.502283,1000,2189,260,0.118776,1000,prediction_example
76,guo_readmission,raw,RETAIN_lite_numeric,platt,top_1pct_precision,0.625727,0.409091,0.818182,1000,2189,260,0.118776,1000,prediction_example
79,guo_readmission,raw,RETAIN_lite_numeric,platt,top_20pct_precision,0.327847,0.283105,0.374429,1000,2189,260,0.118776,1000,prediction_example


In [25]:
# Compact bootstrap table

main_metric_table = (
    sequence_numeric_bootstrap_ci_df[
        sequence_numeric_bootstrap_ci_df["metric"].isin(["auroc", "auprc", "brier", "logloss"])
    ]
    .copy()
)

main_metric_table["mean_ci"] = main_metric_table.apply(
    lambda r: f'{r["mean"]:.3f} [{r["ci_low"]:.3f}, {r["ci_high"]:.3f}]',
    axis=1,
)

main_metric_table = (
    main_metric_table
    .pivot_table(
        index=["task", "version", "model"],
        columns="metric",
        values="mean_ci",
        aggfunc="first",
    )
    .reset_index()
)

display(main_metric_table)


topk_table = (
    sequence_numeric_bootstrap_ci_df[
        sequence_numeric_bootstrap_ci_df["metric"].str.contains("top_")
    ]
    .copy()
)

topk_table["mean_ci"] = topk_table.apply(
    lambda r: f'{r["mean"]:.3f} [{r["ci_low"]:.3f}, {r["ci_high"]:.3f}]',
    axis=1,
)

topk_table = (
    topk_table
    .pivot_table(
        index=["task", "version", "model"],
        columns="metric",
        values="mean_ci",
        aggfunc="first",
    )
    .reset_index()
)

display(topk_table)

metric,task,version,model,auprc,auroc,brier,logloss
0,guo_icu,compressed_condition_era,GRU_2L_numeric,"0.180 [0.114, 0.261]","0.770 [0.720, 0.820]","0.038 [0.031, 0.045]","0.158 [0.133, 0.183]"
1,guo_icu,compressed_dedup,RETAIN_lite_numeric,"0.166 [0.103, 0.238]","0.753 [0.698, 0.803]","0.038 [0.031, 0.045]","0.156 [0.132, 0.181]"
2,guo_icu,compressed_first_last,GRU_1L_numeric,"0.188 [0.122, 0.273]","0.713 [0.644, 0.779]","0.038 [0.031, 0.045]","0.155 [0.131, 0.181]"
3,guo_icu,compressed_hybrid,GRU_2L_numeric,"0.222 [0.144, 0.317]","0.794 [0.740, 0.839]","0.038 [0.031, 0.045]","0.158 [0.132, 0.184]"
4,guo_icu,raw,GRU_1L_numeric,"0.187 [0.114, 0.273]","0.724 [0.666, 0.784]","0.038 [0.031, 0.045]","0.157 [0.132, 0.182]"
5,guo_readmission,compressed_condition_era,RETAIN_lite_numeric,"0.351 [0.292, 0.413]","0.755 [0.720, 0.787]","0.092 [0.084, 0.101]","0.317 [0.294, 0.343]"
6,guo_readmission,compressed_dedup,RETAIN_lite_numeric,"0.380 [0.319, 0.445]","0.776 [0.742, 0.807]","0.088 [0.080, 0.097]","0.305 [0.282, 0.329]"
7,guo_readmission,compressed_first_last,RETAIN_lite_numeric,"0.374 [0.314, 0.439]","0.772 [0.736, 0.805]","0.090 [0.082, 0.099]","0.309 [0.287, 0.333]"
8,guo_readmission,compressed_hybrid,RETAIN_lite_numeric,"0.366 [0.311, 0.428]","0.777 [0.745, 0.807]","0.092 [0.083, 0.100]","0.314 [0.291, 0.338]"
9,guo_readmission,raw,RETAIN_lite_numeric,"0.382 [0.320, 0.446]","0.772 [0.738, 0.804]","0.089 [0.081, 0.098]","0.307 [0.285, 0.332]"


metric,task,version,model,top_10pct_precision,top_1pct_precision,top_20pct_precision,top_5pct_precision
0,guo_icu,compressed_condition_era,GRU_2L_numeric,"0.133 [0.088, 0.181]","0.356 [0.143, 0.571]","0.120 [0.088, 0.157]","0.213 [0.137, 0.294]"
1,guo_icu,compressed_dedup,RETAIN_lite_numeric,"0.161 [0.113, 0.216]","0.298 [0.095, 0.524]","0.115 [0.083, 0.147]","0.191 [0.118, 0.265]"
2,guo_icu,compressed_first_last,GRU_1L_numeric,"0.182 [0.127, 0.240]","0.371 [0.189, 0.571]","0.112 [0.081, 0.147]","0.259 [0.176, 0.353]"
3,guo_icu,compressed_hybrid,GRU_2L_numeric,"0.168 [0.118, 0.221]","0.460 [0.238, 0.714]","0.129 [0.098, 0.164]","0.188 [0.108, 0.275]"
4,guo_icu,raw,GRU_1L_numeric,"0.161 [0.113, 0.216]","0.356 [0.143, 0.571]","0.109 [0.081, 0.142]","0.205 [0.127, 0.285]"
5,guo_readmission,compressed_condition_era,RETAIN_lite_numeric,"0.403 [0.338, 0.470]","0.574 [0.318, 0.773]","0.309 [0.267, 0.358]","0.520 [0.418, 0.618]"
6,guo_readmission,compressed_dedup,RETAIN_lite_numeric,"0.449 [0.384, 0.521]","0.600 [0.364, 0.818]","0.347 [0.297, 0.397]","0.512 [0.409, 0.609]"
7,guo_readmission,compressed_first_last,RETAIN_lite_numeric,"0.436 [0.370, 0.502]","0.594 [0.364, 0.818]","0.331 [0.288, 0.377]","0.474 [0.373, 0.573]"
8,guo_readmission,compressed_hybrid,RETAIN_lite_numeric,"0.379 [0.315, 0.447]","0.662 [0.455, 0.864]","0.315 [0.274, 0.363]","0.500 [0.391, 0.609]"
9,guo_readmission,raw,RETAIN_lite_numeric,"0.431 [0.370, 0.502]","0.626 [0.409, 0.818]","0.328 [0.283, 0.374]","0.528 [0.427, 0.627]"


## 11. Upload artifacts to ClearML

In [17]:
if task is not None:
    task.upload_artifact("sequence_numeric_results", sequence_numeric_results_df)
    task.upload_artifact("sequence_numeric_topk", sequence_numeric_topk_df)
    task.upload_artifact("numeric_coverage_sequence_length", numeric_coverage_df)
    task.upload_artifact("results_dir", str(RESULTS_DIR))
    print("Uploaded artifacts to ClearML")
else:
    print("ClearML disabled; artifacts are saved locally in", RESULTS_DIR)

ClearML disabled; artifacts are saved locally in ehrshot_sequence_numeric_results


In [26]:
tabular_reference_df = pd.DataFrame([
    {
        "task": "guo_readmission",
        "reference": "tabular HistGradientBoosting + Platt",
        "auroc": 0.764,
        "auprc": 0.373,
        "brier": 0.090,
        "logloss": 0.311,
    },
    {
        "task": "guo_icu",
        "reference": "tabular RandomForest + Platt",
        "auroc": 0.794,
        "auprc": 0.190,
        "brier": 0.038,
        "logloss": 0.150,
    },
])

numeric_reference_df = pd.DataFrame([
    {
        "task": "guo_readmission",
        "reference": "numeric sequence raw + RETAIN_lite_numeric",
        "auroc": 0.772,
        "auprc": 0.380,
        "brier": 0.089,
        "logloss": 0.307,
    },
    {
        "task": "guo_icu",
        "reference": "numeric sequence compressed_hybrid + GRU_2L_numeric",
        "auroc": 0.794,
        "auprc": 0.216,
        "brier": 0.038,
        "logloss": 0.158,
    },
])

display(pd.concat([tabular_reference_df, numeric_reference_df], ignore_index=True))

,task,reference,auroc,auprc,brier,logloss
0,guo_readmission,tabular HistGradientBoosting + Platt,0.764,0.373,0.090,0.311
1,guo_icu,tabular RandomForest + Platt,0.794,0.190,0.038,0.150
2,guo_readmission,numeric sequence raw + RETAIN_lite_numeric,0.772,0.380,0.089,0.307
3,guo_icu,numeric sequence compressed_hybrid + GRU_2L_nu...,0.794,0.216,0.038,0.158


## Главные выводы

1. Числовой сигнал присутствует примерно в 69–72% событий, так что он не маргинальный.
2. Для `guo_readmission` модели с числовыми признаками не показали явного улучшения по сравнению с предыдущим code-only sequence baseline. Лучшая числовая конфигурация была близка к code-only, но не превзошла лучший code-only `compressed_dedup + RETAIN_lite`.
3. Для `guo_icu` числовые sequence-модели существенно улучшились по сравнению с code-only sequence baseline. Лучшая конфигурация была `compressed_hybrid + GRU_2L_numeric`, с AUROC 0.794 и AUPRC 0.216.
4. Числовые sequence-модели почти догнали табличный ICU baseline по AUROC и превзошли его по AUPRC, но LogLoss остался немного хуже.
5. Compression по-прежнему снижает длину последовательности только незначительно, потому что большинство событий EHRSHOT — это измерения / числовые события, а не диагнозы.
6. Следующий шаг — комбинированная модель: табличные агрегированные признаки + sequence embedding.